# GSC Traffic Forecaster — TimesFM + Gemini

## ⚠️ HOW TO RUN THIS NOTEBOOK

This notebook is split into **two sections** separated by a required restart.

**First time setup (do this once):**
1. Go to **Runtime → Change runtime type** and select **T4 GPU**
2. Run **Section 1 only** (the Install cell below)
3. When it prints `✅ Install complete — RESTART YOUR RUNTIME NOW`, go to **Runtime → Restart session**
4. After restart, run **all remaining cells** from Section 2 onward

**Subsequent runs (runtime already set up):**
- Skip Section 1 entirely. Run from Section 2 onward.

---
# SECTION 1 — Install Dependencies
### Run this cell once, then restart your runtime.

In [ ]:
import os, shutil, sys

# Clean up any shadowing folders
for folder in ['/content/timesfm', '/content/google']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f'Removed: {folder}')

os.chdir('/content')

print('Step 1/3: Installing TimesFM from GitHub source...')
!pip install -q git+https://github.com/google-research/timesfm.git

print('Step 2/3: Installing remaining dependencies...')
!pip install -q google-genai openpyxl huggingface_hub pynarrative

print('Step 3/3: Verifying...')
import subprocess, sys as _sys
r = subprocess.run(
    [_sys.executable, '-c', 'import timesfm; print("TimesFM installed")'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print('Verification complete.')
    print('\n' + '=' * 60)
    print('INSTALL COMPLETE - RESTART YOUR RUNTIME NOW')
    print('Runtime -> Restart session')
    print('Then skip this cell and run all cells from Section 2.')
    print('=' * 60)
else:
    print('Verification failed:', r.stderr[-400:])

---
# SECTION 2 — Configuration
### Edit the variables in this cell before running anything else.

In [ ]:
import glob
import os

# ── AUTO-DETECT GSC FILE ───────────────────────────────────────────
xlsx_files = glob.glob('/content/*.xlsx')

if len(xlsx_files) == 0:
    raise FileNotFoundError(
        "No .xlsx file found in /content/. Please upload your GSC export and re-run."
    )
elif len(xlsx_files) == 1:
    GSC_FILE = xlsx_files[0]
    print(f"✅ Auto-detected file: {GSC_FILE}")
else:
    print("Multiple .xlsx files found. Please choose one:\n")
    for i, f in enumerate(xlsx_files):
        print(f"  [{i}] {os.path.basename(f)}")
    choice = int(input("\nEnter the number of the file to use: "))
    GSC_FILE = xlsx_files[choice]
    print(f"\nUsing: {GSC_FILE}")

# ── REMAINING CONFIGURATION ───────────────────────────────────────

# Forecast horizon in days (365 = ~12 months)
FORECAST_DAYS = 365

# Number of URLs to send per Gemini classification batch
BATCH_SIZE = 50

# Multipliers for what-if scenarios (update after Section 3)
SCENARIO_ADJUSTMENTS = {
    # 'Blog/Learn':  {'clicks': 1.10, 'impressions': 1.10, 'ctr': 1.0},
}

print('✅ Configuration loaded.')

---
# SECTION 3 — URL Classification with Gemini
### Gemini will first discover your site's taxonomy, then classify all URLs.

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import re
from google.colab import userdata
from google import genai
from google.genai import types

# ── Load file ────────────────────────────────────────────────────
chart_df = pd.read_excel(GSC_FILE, sheet_name='Chart')
pages_df = pd.read_excel(GSC_FILE, sheet_name='Pages')

# Standardise column names
chart_df.columns = [c.strip() for c in chart_df.columns]
pages_df.columns = [c.strip() for c in pages_df.columns]

# Rename Pages columns robustly
pages_df = pages_df.rename(columns={
    pages_df.columns[0]: 'URL',
    pages_df.columns[1]: 'Clicks',
    pages_df.columns[2]: 'Impressions',
    pages_df.columns[3]: 'CTR',
    pages_df.columns[4]: 'Position'
})

chart_df['Date'] = pd.to_datetime(chart_df['Date'])
chart_df = chart_df.sort_values('Date').reset_index(drop=True)

print(f'Chart sheet: {len(chart_df)} daily rows')
print(f'  Date range: {chart_df["Date"].min().date()} → {chart_df["Date"].max().date()}')
print(f'Pages sheet: {len(pages_df)} URLs')
print(f'\nChart columns: {list(chart_df.columns)}')
print(f'Pages columns: {list(pages_df.columns)}')

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import re
from google.colab import userdata
from google import genai
from google.genai import types
from google.api_core.exceptions import ClientError # Corrected import for ClientError

# ── Load file ────────────────────────────────────────────────────
chart_df = pd.read_excel(GSC_FILE, sheet_name='Chart')
pages_df = pd.read_excel(GSC_FILE, sheet_name='Pages')

# Standardise column names
chart_df.columns = [c.strip() for c in chart_df.columns]
pages_df.columns = [c.strip() for c in pages_df.columns]

# Rename Pages columns robustly
pages_df = pages_df.rename(columns={
    pages_df.columns[0]: 'URL',
    pages_df.columns[1]: 'Clicks',
    pages_df.columns[2]: 'Impressions',
    pages_df.columns[3]: 'CTR',
    pages_df.columns[4]: 'Position'
})

chart_df['Date'] = pd.to_datetime(chart_df['Date'])
chart_df = chart_df.sort_values('Date').reset_index(drop=True)

print(f'Chart sheet: {len(chart_df)} daily rows')
print(f'  Date range: {chart_df['Date'].min().date()} → {chart_df['Date'].max().date()}')
print(f'Pages sheet: {len(pages_df)} URLs')
print(f'\nChart columns: {list(chart_df.columns)}')
print(f'Pages columns: {list(pages_df.columns)}')


# ── Configure Gemini client ───────────────────────────────────────
api_key = userdata.get('GOOGLE_API_KEY')
gemini = genai.Client(api_key=api_key)
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'

all_urls = pages_df['URL'].dropna().astype(str).tolist()

# ── PASS 1: Taxonomy Discovery ────────────────────────────────────
# Send a sample of URLs and ask Gemini to infer the site's page types

sample_urls = all_urls[:150]  # Sample for taxonomy discovery
url_sample_text = '\n'.join(sample_urls)

taxonomy_prompt = f"""You are an SEO analyst. Analyze the following list of URLs from a single website.

Your task:
1. Identify the distinct page types / sections that exist on this site based on URL patterns.
2. Return a JSON object with two keys:
   - "taxonomy": a list of category name strings (e.g. ["Homepage", "Blog", "Pricing"])
   - "reasoning": a brief explanation of how you identified each category from the URL patterns

Rules:
- Be specific to THIS site's architecture, not generic
- Aim for 4-10 meaningful categories — not too broad, not too granular
- Every URL must be classifiable into one of your categories
- Include an "Other" category as a catch-all
- Return ONLY valid JSON. No markdown, no backticks, no preamble.

URLs to analyze:
{url_sample_text}"""

print('Pass 1: Discovering site taxonomy...')

taxonomy_response = None
for attempt in range(5): # Retry up to 5 times
    try:
        taxonomy_response = gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=taxonomy_prompt
        )
        break # If successful, break the loop
    except ClientError as e:
        if e.status_code == 429: # Resource Exhausted
            print(f'  ⚠️  Attempt {attempt+1} failed due to quota exhaustion. Retrying in {2**(attempt+1)} seconds...')
            time.sleep(2**(attempt+1)) # Exponential backoff
        else:
            raise # Re-raise other ClientErrors
    except Exception as e:
        print(f'  ⚠️  Attempt {attempt+1} failed with unexpected error: {e}. Retrying in {2**(attempt+1)} seconds...')
        time.sleep(2**(attempt+1))

if taxonomy_response is None:
    raise Exception('Failed to discover taxonomy after multiple retries.')

raw = taxonomy_response.text.strip()
# Strip markdown code fences if present
raw = re.sub(r'^```(?:json)?\s*', '', raw)
raw = re.sub(r'\s*```$', '', raw)

taxonomy_result = json.loads(raw)
TAXONOMY = taxonomy_result['taxonomy']

print(f'\n✅ Taxonomy discovered: {TAXONOMY}')
print(f'\nGemini reasoning:')
print(taxonomy_result['reasoning'])

In [ ]:
# ── PASS 2: Batch Classification ─────────────────────────────────

def classify_batch(url_batch, taxonomy):
    """Send a batch of URLs to Gemini and return a list of classifications."""
    taxonomy_str = ', '.join(f'"{t}"' for t in taxonomy)
    url_list_text = '\n'.join([f'{i+1}. {u}' for i, u in enumerate(url_batch)])

    prompt = f"""You are an SEO analyst classifying URLs.

Valid categories (use ONLY these exact strings): [{taxonomy_str}]

Classify each URL below. Return a JSON array with one string per URL, in the same order.
Example for 3 URLs: ["Blog", "Homepage", "Pricing"]
Return ONLY the JSON array. No markdown, no backticks, no explanation.

URLs:
{url_list_text}"""

    for attempt in range(5): # Increase retry attempts
        try:
            response = gemini.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt
            )
            raw = response.text.strip()
            raw = re.sub(r'^```(?:json)?\s*', '', raw)
            raw = re.sub(r'\s*```$', '', raw)
            result = json.loads(raw)
            # Validate length matches
            if len(result) == len(url_batch):
                return result
            else:
                print(f'  ⚠️  Length mismatch ({len(result)} vs {len(url_batch)}), retrying...')
                time.sleep(2**(attempt+1)) # Exponential backoff for retries
        except ClientError as e:
            if e.status_code == 429: # Resource Exhausted
                print(f'  ⚠️  Attempt {attempt+1} failed due to quota exhaustion. Retrying in {2**(attempt+1)} seconds...')
                time.sleep(2**(attempt+1)) # Exponential backoff
            else:
                print(f'  ⚠️  Attempt {attempt+1} failed with ClientError: {e}. Retrying in {2**(attempt+1)} seconds...')
                time.sleep(2**(attempt+1)) # Exponential backoff
        except Exception as e:
            print(f'  ⚠️  Attempt {attempt+1} failed with unexpected error: {e}. Retrying in {2**(attempt+1)} seconds...')
            time.sleep(2**(attempt+1))

    # Fallback: return 'Other' for all in this batch if all retries fail
    print(f'  ❌  Failed to classify batch after multiple retries. Assigning "Other" to all URLs in this batch.')
    return ['Other'] * len(url_batch)


# Run classification in batches
all_classifications = []
total_batches = (len(all_urls) + BATCH_SIZE - 1) // BATCH_SIZE

print(f'Pass 2: Classifying {len(all_urls)} URLs in {total_batches} batches of {BATCH_SIZE}...')

for i in range(0, len(all_urls), BATCH_SIZE):
    batch = all_urls[i:i + BATCH_SIZE]
    batch_num = (i // BATCH_SIZE) + 1
    print(f'  Batch {batch_num}/{total_batches}...', end=' ')
    classifications = classify_batch(batch, TAXONOMY)
    all_classifications.extend(classifications)
    print(f'done')
    time.sleep(1)  # Increased gentle rate limiting between batches

pages_df['Page Group'] = all_classifications

print(f'\n✅ Classification complete.')
print('\nPage Group Distribution:')
print(pages_df['Page Group'].value_counts().to_string())

In [ ]:
# ── Review classifications ────────────────────────────────────────
# Check a sample from each group to validate accuracy

print('Sample URLs per Page Group (top 3 each):')
print('=' * 70)
for group in sorted(pages_df['Page Group'].unique()):
    subset = pages_df[pages_df['Page Group'] == group]
    print(f'\n{group} ({len(subset)} URLs):')
    for url in subset['URL'].head(3).tolist():
        print(f'  {url}')

print('\n' + '=' * 70)
print('If any classifications look wrong, you can manually correct them below.')
print('Run the override cell, then proceed to Section 4.')

In [ ]:
import pandas as pd

# ── OPTIONAL: Manual overrides ────────────────────────────────────
# Add URL substring → correct group mappings if needed.
# Leave empty if classifications look correct.

MANUAL_OVERRIDES = {
    '/expert-advice/': 'Resources & Education',
    # '/p/': 'Product Details',
}

for pattern, correct_group in MANUAL_OVERRIDES.items():
    if pattern: # Only apply if pattern is not an empty string
        mask = pages_df['URL'].str.contains(pattern, na=False)
        count = mask.sum()
        pages_df.loc[mask, 'Page Group'] = correct_group
        print(f'Override applied: {count} URLs matching "{pattern}" → "{correct_group}"')

if not MANUAL_OVERRIDES:
    print('No overrides defined. Proceeding with Gemini classifications.')

# Save classified pages for reference
pages_df.to_csv('classified_pages.csv', index=False)
print('\n✅ classified_pages.csv saved to Colab Files with consistent naming.')

---
# SECTION 4 — TimesFM Baseline Forecast
### Loads the model and generates a baseline forecast from historical daily clicks.

In [ ]:
import torch
import numpy as np
import timesfm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

torch.set_float32_matmul_precision('high')

print('Loading TimesFM 2.5 model...')

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    'google/timesfm-2.5-200m-pytorch',
    torch_compile=False  # set True after confirming it loads
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=FORECAST_DAYS,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        fix_quantile_crossing=True,
    )
)

print('✅ TimesFM 2.5 model loaded.')

In [ ]:
history = chart_df['Clicks'].values.astype(float)
last_date = chart_df['Date'].iloc[-1]

print(f'Forecasting {FORECAST_DAYS} days from {last_date.date()}...')

# The model provides 10 quantiles: 10th, 20th, ..., 90th by default.
# Note: TimesFM 2.5 typically outputs 10 specific quantiles.
point_forecast, quantile_forecast = model.forecast(
    horizon=FORECAST_DAYS,
    inputs=[history],
)

# Selecting the 5th percentile (index 0) and 95th percentile (index -1)
# assuming the model is configured for these specific output heads.
baseline_forecast = point_forecast[0]
lower_ci = quantile_forecast[0, :, 0]   # 5th percentile
upper_ci = quantile_forecast[0, :, -1]  # 95th percentile

forecast_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=FORECAST_DAYS,
    freq='D'
)

baseline_df = pd.DataFrame({
    'Date': forecast_dates,
    'Forecast_Clicks': baseline_forecast,
    'Lower_CI': lower_ci,
    'Upper_CI': upper_ci,
})

total_baseline = baseline_df['Forecast_Clicks'].sum()
print(f'✅ Baseline forecast complete.')
print(f'   Projected total clicks over {FORECAST_DAYS} days: {total_baseline:,.0f}')
print(f'   Note: Range set to 5th and 95th percentiles.')

In [ ]:
!pip install -q pynarrative
import pynarrative as pn
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# ── Plot baseline forecast ────────────────────────────────────────

fig, ax = plt.subplots(figsize=(16, 7))

# Historical
ax.plot(chart_df['Date'], chart_df['Clicks'],
        color='#2563EB', linewidth=1.5, label='Historical Clicks', alpha=0.9)

# Forecast
ax.plot(baseline_df['Date'], baseline_df['Forecast_Clicks'],
        color='#DC2626', linewidth=2, linestyle='--', label='Baseline Forecast')

# Confidence interval
ax.fill_between(baseline_df['Date'], baseline_df['Lower_CI'], baseline_df['Upper_CI'],
                color='#DC2626', alpha=0.15, label='80% Confidence Interval')

# Divider line
ax.axvline(x=last_date, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax.text(last_date, ax.get_ylim()[1] * 0.95, ' Forecast →', color='gray', fontsize=10)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)

ax.set_title('GSC Clicks: Historical + 12-Month Baseline Forecast', fontsize=15, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Clicks')
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('baseline_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Historical Narrative Summary ──────────────────────────────────
history_narrative = f"Historical Analysis: From {chart_df['Date'].min().date()} to {chart_df['Date'].max().date()}, the site earned {chart_df['Clicks'].sum():,} clicks, averaging {chart_df['Clicks'].mean():,.0f} daily. Peak traffic reached {chart_df['Clicks'].max():,} on {chart_df.loc[chart_df['Clicks'].idxmax(), 'Date'].date()}."
print('\n' + '='*30 + ' NARRATIVE SUMMARY ' + '='*30)
print(history_narrative)

In [ ]:
import pandas as pd

# Compute page group traffic share (moved from cell _R_-VZF2-uuO)
total_clicks_all = pages_df['Clicks'].sum()

group_summary = pages_df.groupby('Page Group').agg(
    Clicks=('Clicks', 'sum'),
    Impressions=('Impressions', 'sum'),
    Avg_Position=('Position', 'mean'),
    URL_Count=('URL', 'count')
).reset_index()

# Calculate Average CTR for each group
group_summary['Avg_CTR'] = group_summary['Clicks'] / group_summary['Impressions']

group_summary['Traffic_Share'] = group_summary['Clicks'] / total_clicks_all


# Prepare the display version of the summary
display_df = group_summary[[
    'Page Group', 'URL_Count', 'Clicks', 'Impressions', 'Avg_CTR', 'Avg_Position', 'Traffic_Share'
]].sort_values('Clicks', ascending=False).copy()

print('Page Group Summary (Historical Data):')

# Apply styling for better readability in the notebook
styled_df = display_df.style.format({
    'Clicks': '{:,.0f}',
    'Impressions': '{:,.0f}',
    'Avg_CTR': '{:.2%}',
    'Avg_Position': '{:.1f}',
    'Traffic_Share': '{:.1%}'
}).background_gradient(subset=['Clicks'], cmap='BuGn')

display(styled_df)

---
# SECTION 5 — What-If Scenario Analysis

### Before running this section:
1. Look at the **Page Group Distribution** printed in Section 3
2. Fill in `SCENARIO_ADJUSTMENTS` with your desired multipliers
3. Re-run the Configuration cell, then run Section 5

**How it works:** Each page group's share of total traffic is calculated from the Pages sheet.
Your multipliers scale that group's contribution to the forecast up or down.

In [ ]:
import glob
import os

# ── AUTO-DETECT GSC FILE ───────────────────────────────────────────
xlsx_files = glob.glob('/content/*.xlsx')

if len(xlsx_files) == 0:
    raise FileNotFoundError(
        "No .xlsx file found in /content/. Please upload your GSC export and re-run."
    )
elif len(xlsx_files) == 1:
    GSC_FILE = xlsx_files[0]
    print(f"✅ Auto-detected file: {GSC_FILE}")
else:
    print("Multiple .xlsx files found. Please choose one:\n")
    for i, f in enumerate(xlsx_files):
        print(f"  [{i}] {os.path.basename(f)}")
    choice = int(input("\nEnter the number of the file to use: "))
    GSC_FILE = xlsx_files[choice]
    print(f"\nUsing: {GSC_FILE}")

# ── REMAINING CONFIGURATION ───────────────────────────────────────

# Forecast horizon in days (365 = ~12 months)
FORECAST_DAYS = 365

# Number of URLs to send per Gemini classification batch
BATCH_SIZE = 50

# ── ADVANCED SCENARIO ADJUSTMENTS ─────────────────────────────────
# Lever options:
# - 'impressions': multiplier (1.10 = +10% volume)
# - 'ctr': flat multiplier for CTR
# - 'target_rank': integer 1-20 (uses interpolated CTR curve)

SCENARIO_ADJUSTMENTS = {
    'Category Browse':  {'target_rank': 5.5, 'impressions': 0.00, 'ctr': 0.00},
    'Resources & Education': {'target_rank': 5, 'impressions': 1.00}
}

print('✅ Configuration updated with correct taxonomy keys.')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Prepare data: Focus on top 20 positions where CTR is most meaningful
# We use the 'pages_df' which already has 'Page Group' from the Gemini step
curve_df = pages_df[(pages_df['Position'] <= 20) & (pages_df['Impressions'] > 0)].copy()

# 2. Round position to nearest integer for grouping (e.g., 1.2 -> 1, 1.8 -> 2)
curve_df['Rank'] = curve_df['Position'].round(0).astype(int)

# 3. Calculate Weighted Average CTR per Group and Rank
# Weighted average (Sum of Clicks / Sum of Impressions) is better than a simple mean of CTRs
ctr_group_stats = curve_df.groupby(['Page Group', 'Rank']).agg(
    Total_Clicks=('Clicks', 'sum'),
    Total_Imps=('Impressions', 'sum')
).reset_index()

ctr_group_stats['Avg_CTR'] = ctr_group_stats['Total_Clicks'] / ctr_group_stats['Total_Imps']

# 4. Visualization
plt.figure(figsize=(14, 8))
sns.lineplot(data=ctr_group_stats, x='Rank', y='Avg_CTR', hue='Page Group', marker='o', linewidth=2)

plt.title('GSC CTR Curve: Average CTR vs Position by Page Group', fontsize=16, fontweight='bold')
plt.xlabel('Average Position (Rounded)', fontsize=12)
plt.ylabel('Click-Through Rate (CTR)', fontsize=12)
plt.xticks(range(1, 21))
plt.grid(True, alpha=0.2)
plt.legend(title='Page Group', bbox_to_anchor=(1.05, 1), loc='upper left')

# Format Y axis as percentage
from matplotlib.ticker import PercentFormatter
plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))

plt.tight_layout()
plt.show()

# 5. Display a pivot table for quick reference
print('\nCTR Reference Table (Positions 1-10):')
pivot_table = ctr_group_stats[ctr_group_stats['Rank'] <= 10].pivot(index='Rank', columns='Page Group', values='Avg_CTR')
display(pivot_table.style.format('{:.2%}'))

In [ ]:
# ── Compute page group traffic share ─────────────────────────────

total_clicks_all = pages_df['Clicks'].sum()

group_summary = pages_df.groupby('Page Group').agg(
    Clicks=('Clicks', 'sum'),
    Impressions=('Impressions', 'sum'),
    Avg_Position=('Position', 'mean'),
    URL_Count=('URL', 'count')
).reset_index()

# Calculate Average CTR for each group
group_summary['Avg_CTR'] = group_summary['Clicks'] / group_summary['Impressions']

group_summary['Traffic_Share'] = group_summary['Clicks'] / total_clicks_all

print('Page Group Summary (from historical data):')
print('=' * 90)

# Prepare and style the DataFrame for display
display_df = group_summary[[
    'Page Group', 'URL_Count', 'Clicks', 'Impressions', 'Avg_CTR', 'Avg_Position', 'Traffic_Share'
]].sort_values('Clicks', ascending=False).copy()

styled_df = display_df.style.format({
    'Clicks': '{:,.0f}',
    'Impressions': '{:,.0f}',
    'Avg_CTR': '{:.2%}',
    'Avg_Position': '{:.1f}',
    'Traffic_Share': '{:.1%}'
}).background_gradient(subset=['Clicks'], cmap='BuGn')

display(styled_df)

if not SCENARIO_ADJUSTMENTS:
    print('\n⚠️  SCENARIO_ADJUSTMENTS is empty.')
    print('   Go to the Configuration cell (Section 2), fill it in, re-run that cell, then re-run Section 5.')

In [ ]:
import pandas as pd
import numpy as np

# 1. Create a complete grid of all Page Groups and all Ranks (1-20)
all_groups = ctr_group_stats['Page Group'].unique()
all_ranks = range(1, 21)
full_grid = pd.MultiIndex.from_product([all_groups, all_ranks], names=['Page Group', 'Rank']).to_frame(index=False)

# 2. Merge with our existing stats
final_curve = full_grid.merge(ctr_group_stats[['Page Group', 'Rank', 'Avg_CTR']], on=['Page Group', 'Rank'], how='left')

# 3. Interpolate gaps within each group
# This fills Rank 2 if you have Rank 1 and 3
final_curve['Interpolated_CTR'] = final_curve.groupby('Page Group')['Avg_CTR'].transform(lambda x: x.interpolate(limit_direction='both'))

# 4. Fallback: If a group is missing a rank entirely (e.g. no Rank 1), use the site-wide average for that rank
site_avg_ctr = ctr_group_stats.groupby('Rank')['Avg_CTR'].mean()
final_curve['Interpolated_CTR'] = final_curve.apply(
    lambda row: site_avg_ctr[row['Rank']] if pd.isna(row['Interpolated_CTR']) else row['Interpolated_CTR'],
    axis=1
)

# 5. Visualize the 'Smoothed' Curves
plt.figure(figsize=(14, 8))
sns.lineplot(data=final_curve, x='Rank', y='Interpolated_CTR', hue='Page Group', marker='x', linestyle='--')
plt.title('Smoothed CTR Curve (Interpolated & Site-Avg Fallback)', fontsize=15)
plt.xticks(range(1, 21))
plt.ylabel('CTR')
plt.grid(True, alpha=0.2)
plt.show()

print("Filled CTR Table (No more NaNs):")
display(final_curve[final_curve['Rank'] <= 10].pivot(index='Rank', columns='Page Group', values='Interpolated_CTR').style.format('{:.2%}'))

In [ ]:
import pandas as pd
import numpy as np

# ── ROADMAP CONFIGURATION ──────────────────────────────────────────
# How many implementation 'events' or projects are planned?
NUM_IMPLEMENTATIONS = 4

# When does each implementation start? (Relative to forecast start)
# Example: [0, 91, 181, 271] for one project per quarter
START_DAYS = [0, 91, 181, 271]

# Duration of the ramp-up per implementation
RAMP_DURATION = 60

print(f'Roadmap set: {NUM_IMPLEMENTATIONS} projects starting on days {START_DAYS}')

In [ ]:
def calculate_phased_multipliers(total_target_mult, num_impl, start_days, ramp_len, forecast_len):
    """Calculates a composite multiplier array for multiple ramp-up phases."""
    growth_per_project = (total_target_mult - 1.0) / num_impl
    composite_multipliers = np.ones(forecast_len)

    for start_day in start_days:
        days_since_start = np.arange(forecast_len) - start_day
        project_ramp = np.clip(days_since_start / ramp_len, 0, 1)
        composite_multipliers += (project_ramp * growth_per_project)
    return composite_multipliers

# ── ENSURE BASE MULTIPLIER EXISTS ──────────────────────────────────
if 'target_site_multiplier' not in globals():
    print('🔄 Calculating base scenario multiplier first...')
    target_site_multiplier = 0.0
    for _, row in group_summary.iterrows():
        group = row['Page Group']
        share = row['Traffic_Share']
        adj = SCENARIO_ADJUSTMENTS.get(group, {})
        imp_mult = adj.get('impressions', 1.0)
        target_rank = adj.get('target_rank')
        if target_rank and 'final_curve' in globals():
            try:
                target_ctr = final_curve[(final_curve['Page Group'] == group) & (final_curve['Rank'] == int(round(target_rank)))]['Interpolated_CTR'].values[0]
                current_ctr = row['Clicks'] / row['Impressions'] if row['Impressions'] > 0 else 0.02
                ctr_mult = target_ctr / current_ctr if current_ctr > 0 else 1.0
            except: ctr_mult = adj.get('ctr', 1.0)
        else: ctr_mult = adj.get('ctr', 1.0)
        target_site_multiplier += (share * imp_mult * ctr_mult)

# ── GENERATE PHASEED ROADMAP ───────────────────────────────────────
phased_multipliers = calculate_phased_multipliers(
    target_site_multiplier,
    NUM_IMPLEMENTATIONS,
    START_DAYS,
    RAMP_DURATION,
    len(forecast_dates)
)

roadmap_forecast = baseline_forecast * phased_multipliers
roadmap_lower = lower_ci * phased_multipliers
roadmap_upper = upper_ci * phased_multipliers

roadmap_df = pd.DataFrame({
    'Date': forecast_dates,
    'Baseline': baseline_forecast,
    'Phased_Scenario': roadmap_forecast,
    'Lower_CI': roadmap_lower,
    'Upper_CI': roadmap_upper,
    'Multiplier': phased_multipliers
})

delta_roadmap = roadmap_df['Phased_Scenario'].sum() - roadmap_df['Baseline'].sum()
print(f'✅ Phased Roadmap Calculated.')
print(f'   Annual Roadmap Impact: {delta_roadmap:,.0f} extra clicks.')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── Setup Figure with Two Subplots ────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12), gridspec_kw={'height_ratios': [1, 1]})

# ── PLOT 1: Site-Wide Phased Roadmap ───────────────────────────────
ax1.plot(roadmap_df['Date'], roadmap_df['Baseline'], color='gray', linestyle='--', alpha=0.5, label='Baseline')
ax1.plot(roadmap_df['Date'], roadmap_df['Phased_Scenario'], color='#8B5CF6', linewidth=3, label='Phased Roadmap (Site-Wide)')

# Visual markers for project starts
for day in START_DAYS:
    if day < len(forecast_dates):
        event_date = roadmap_df['Date'].iloc[day]
        ax1.axvline(x=event_date, color='#C4B5FD', linestyle=':', alpha=0.8)
        ax1.text(event_date, ax1.get_ylim()[1]*0.05, ' Project Start', rotation=90, color='#7C3AED', fontsize=9)

ax1.fill_between(roadmap_df['Date'], roadmap_df['Lower_CI'], roadmap_df['Upper_CI'], color='#8B5CF6', alpha=0.1)
ax1.set_title('SEO Roadmap: Phased Implementation Growth (Site-Wide)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Daily Clicks')
ax1.legend()
ax1.grid(True, alpha=0.15)

# ── PLOT 2: Segment Specific (Category Browse) ──────────────────────
target_group = 'Category Browse'
group_share = group_summary[group_summary['Page Group'] == target_group]['Traffic_Share'].values[0]

# Calculate historical segment clicks (approximate from daily total * share)
# or use precise segment history if available.
# Here we use the share-weighted total for visual consistency:
hist_segment = chart_df['Clicks'] * group_share

# Apply the phased multiplier to the segment baseline
segment_baseline = baseline_forecast * group_share
segment_roadmap = segment_baseline * phased_multipliers

# Historical Plot
ax2.plot(chart_df['Date'], hist_segment, color='#2563EB', linewidth=1.5, label=f'Historical {target_group}', alpha=0.7)

# Forecast Plot
ax2.plot(roadmap_df['Date'], segment_roadmap, color='#10B981', linewidth=3, label=f'Roadmap {target_group}')

# Add the forecast divider
ax2.axvline(x=chart_df['Date'].max(), color='black', linestyle='-', linewidth=1, alpha=0.4)

ax2.set_title(f'Segment Detail: {target_group} Roadmap Forecast', fontsize=14, fontweight='bold')
ax2.set_ylabel('Daily Clicks')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.legend()
ax2.grid(True, alpha=0.15)

plt.tight_layout()
plt.show()

In [ ]:
# ── Updated Scenario Calculation (365-Day Linear Scaling) ──────────────────

if not SCENARIO_ADJUSTMENTS:
    print('No scenario adjustments defined.')
else:
    target_site_multiplier = 0.0
    for _, row in group_summary.iterrows():
        group = row['Page Group']
        share = row['Traffic_Share']
        adj = SCENARIO_ADJUSTMENTS.get(group, {})

        imp_mult = adj.get('impressions', 1.0)
        target_rank = adj.get('target_rank')

        if target_rank and 'final_curve' in globals():
            try:
                target_ctr = final_curve[(final_curve['Page Group'] == group) & (final_curve['Rank'] == int(round(target_rank)))]['Interpolated_CTR'].values[0]
                current_ctr = row['Clicks'] / row['Impressions'] if row['Impressions'] > 0 else 0.02
                ctr_mult = target_ctr / current_ctr if current_ctr > 0 else 1.0
            except:
                ctr_mult = adj.get('ctr', 1.0)
        else:
            ctr_mult = adj.get('ctr', 1.0)

        target_site_multiplier += (share * imp_mult * ctr_mult)

    # APPLY 365-DAY LINEAR SCALING instead of 60-day ramp
    days_array = np.arange(len(forecast_dates))
    daily_multipliers = 1.0 + (target_site_multiplier - 1.0) * (days_array / 365)

    scenario_forecast = baseline_forecast * daily_multipliers
    scenario_lower = lower_ci * daily_multipliers
    scenario_upper = upper_ci * daily_multipliers

    scenario_df = pd.DataFrame({
        'Date': forecast_dates,
        'Baseline_Clicks': baseline_forecast,
        'Scenario_Clicks': scenario_forecast,
        'Scenario_Lower_CI': scenario_lower,
        'Scenario_Upper_CI': scenario_upper,
        'Daily_Multiplier': daily_multipliers
    })

    delta = scenario_df['Scenario_Clicks'].sum() - scenario_df['Baseline_Clicks'].sum()
    delta_pct = (delta / scenario_df['Baseline_Clicks'].sum()) * 100
    print(f'✅ Site-wide Multiplier (at 12-month maturity): {target_site_multiplier:.4f}')

In [ ]:
# ── Plot ramp-up scenario comparison ──────────────────────────────────────

if not SCENARIO_ADJUSTMENTS:
    print('Define SCENARIO_ADJUSTMENTS first.')
else:
    fig, ax = plt.subplots(figsize=(16, 7))

    ax.plot(chart_df['Date'], chart_df['Clicks'], color='#2563EB', linewidth=1.5, label='Historical', alpha=0.7)
    ax.plot(scenario_df['Date'], scenario_df['Baseline_Clicks'], color='gray', linewidth=1.5, linestyle='--', label='Baseline Forecast', alpha=0.5)
    ax.plot(scenario_df['Date'], scenario_df['Scenario_Clicks'], color='#16A34A', linewidth=2.5, label='Growth Scenario (Linear Ramp)')

    # Highlight the ramp-up period
    ramp_end_date = scenario_df['Date'].iloc[min(FORECAST_DAYS, len(scenario_df)-1)]
    ax.axvspan(last_date, ramp_end_date, color='#FEF3C7', alpha=0.3, label='365-Day Ramp-Up Phase')

    ax.fill_between(scenario_df['Date'], scenario_df['Scenario_Lower_CI'], scenario_df['Scenario_Upper_CI'], color='#16A34A', alpha=0.1, label='80% CI')

    ax.axvline(x=last_date, color='black', linestyle='-', linewidth=1, alpha=0.5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.xticks(rotation=45)

    summary_text = f'Projected Annual Impact:\n{delta:,.0f} extra clicks ({delta_pct:+.1f}%)'
    ax.text(0.02, 0.97, summary_text, transform=ax.transAxes, fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#DCFCE7', alpha=0.9))

    ax.set_title('GSC Forecast: 365-Day Linear Growth Implementation', fontsize=15, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.15)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Monthly rollup table ──────────────────────────────────────────

if not SCENARIO_ADJUSTMENTS:
    print('Define SCENARIO_ADJUSTMENTS first.')
else:
    monthly = scenario_df.copy()
    monthly['Month'] = monthly['Date'].dt.to_period('M')

    monthly_summary = monthly.groupby('Month').agg(
        Baseline=('Baseline_Clicks', 'sum'),
        Scenario=('Scenario_Clicks', 'sum'),
    ).reset_index()

    monthly_summary['Delta'] = monthly_summary['Scenario'] - monthly_summary['Baseline']
    monthly_summary['Delta_%'] = (monthly_summary['Delta'] / monthly_summary['Baseline'] * 100).round(1)

    print('Monthly Forecast Summary (Site-Wide):')
    print('=' * 65)
    print(f'{"Month":<12} {"Baseline":>12} {"Scenario":>12} {"Delta":>12} {"Delta %":>10}')
    print('-' * 65)
    for _, r in monthly_summary.iterrows():
        print(f'{str(r["Month"]):<12} {r["Baseline"]:>12,.0f} {r["Scenario"]:>12,.0f} {r["Delta"]:>+12,.0f} {r["Delta_%"]:>9.1f}%')
    print('=' * 65)
    print(f'{"TOTAL":<12} {monthly_summary["Baseline"].sum():>12,.0f} {monthly_summary["Scenario"].sum():>12,.0f} {monthly_summary["Delta"].sum():>+12,.0f}')

    # Export
    scenario_df.to_csv('scenario_forecast_daily.csv', index=False)
    monthly_summary.to_csv('scenario_forecast_monthly.csv', index=False)
    print("\n✅ Exports saved: scenario_forecast_daily.csv, scenario_forecast_monthly.csv")

In [ ]:
import pandas as pd
import numpy as np
import os

# ── Generate Segment-Specific Monthly Tables (Linear 12-Month) ─────────────────────

if not SCENARIO_ADJUSTMENTS:
    print('Define SCENARIO_ADJUSTMENTS first.')
else:
    for group in SCENARIO_ADJUSTMENTS.keys():
        if group not in group_summary['Page Group'].values:
            print(f'⚠️  Skipping {group}: Not found in taxonomy.')
            continue

        # 1. Calculate Maturity Multiplier for this specific group
        row = group_summary[group_summary['Page Group'] == group].iloc[0]
        share = row['Traffic_Share']
        adj = SCENARIO_ADJUSTMENTS[group]

        imp_mult = adj.get('impressions', 1.0)
        target_rank = adj.get('target_rank')

        if target_rank and 'final_curve' in globals():
            try:
                target_ctr = final_curve[(final_curve['Page Group'] == group) & (final_curve['Rank'] == int(round(target_rank)))]['Interpolated_CTR'].values[0]
                current_ctr = row['Clicks'] / row['Impressions'] if row['Impressions'] > 0 else 0.02
                group_maturity_mult = (imp_mult * target_ctr / current_ctr) if current_ctr > 0 else imp_mult
            except: group_maturity_mult = adj.get('ctr', 1.0) * imp_mult
        else: group_maturity_mult = adj.get('ctr', 1.0) * imp_mult

        # 2. Apply 365-day linear scale to the segment baseline
        days_array = np.arange(len(forecast_dates))
        daily_group_mults = 1.0 + (group_maturity_mult - 1.0) * (days_array / 365)

        seg_baseline = baseline_forecast * share
        seg_scenario = seg_baseline * daily_group_mults

        # 3. Monthly Rollup
        seg_df = pd.DataFrame({
            'Date': forecast_dates,
            'Baseline': seg_baseline,
            'Scenario': seg_scenario
        })
        seg_df['Month'] = seg_df['Date'].dt.to_period('M')

        monthly_seg = seg_df.groupby('Month').agg(
            Baseline=('Baseline', 'sum'),
            Scenario=('Scenario', 'sum')
        ).reset_index()

        monthly_seg['Delta'] = monthly_seg['Scenario'] - monthly_seg['Baseline']
        monthly_seg['Delta_%'] = (monthly_seg['Delta'] / monthly_seg['Baseline'] * 100).round(1)

        # 4. Display and Export
        print(f'\nMonthly Forecast Summary (Linear Scale): {group}')
        print('=' * 65)
        display(monthly_seg.style.format({
            'Baseline': '{:,.0f}',
            'Scenario': '{:,.0f}',
            'Delta': '{:+,.0f}',
            'Delta_%': '{:+.1f}%'
        }))

        safe_name = group.replace(' ', '_').replace('&', 'and').lower()
        csv_path = f'/content/scenario_forecast_{safe_name}.csv'
        monthly_seg.to_csv(csv_path, index=False)
        print(f'✅ Exported: {csv_path}')

In [ ]:
!pip install -q pynarrative

In [ ]:
import pynarrative as pn

# Generate narrative for historical data
historical_stats = {
    'start_date': chart_df['Date'].min().date(),
    'end_date': chart_df['Date'].max().date(),
    'total_clicks': chart_df['Clicks'].sum(),
    'avg_daily_clicks': chart_df['Clicks'].mean(),
    'peak_clicks': chart_df['Clicks'].max(),
    'peak_date': chart_df.loc[chart_df['Clicks'].idxmax(), 'Date'].date()
}

history_narrative = f"Analysis of traffic from {historical_stats['start_date']} to {historical_stats['end_date']} shows a total of {historical_stats['total_clicks']:,} clicks. The site averaged {historical_stats['avg_daily_clicks']:,.0f} clicks per day, with a peak of {historical_stats['peak_clicks']:,} on {historical_stats['peak_date']}."

print("### Historical Traffic Narrative")
print(history_narrative)

In [ ]:
import pandas as pd

# Calculate total_scenario from the existing scenario_df
total_scenario = scenario_df['Scenario_Clicks'].sum()

# Generate narrative for the scenario forecast
scenario_narrative_text = f"The applied SEO scenario, focusing on {list(SCENARIO_ADJUSTMENTS.keys())}, is projected to result in {total_scenario:,.0f} total clicks over the next year. Compared to the baseline forecast of {total_baseline:,.0f}, this represents a change of {delta:,.0f} clicks ({delta_pct:+.1f}%)."

print("### Scenario Forecast Narrative")
print(scenario_narrative_text)

In [ ]:
import pandas as pd
import numpy as np

# ── 12-Month Linear Implementation Roadmap ──────────────────────
target_group = 'Category Browse'
if target_group in group_summary['Page Group'].values:
    # 1. Setup the segment baseline
    share = group_summary[group_summary['Page Group'] == target_group]['Traffic_Share'].values[0]
    seg_baseline_daily = baseline_forecast * share

    # 2. Create the roadmap dataframe
    roadmap_12m = pd.DataFrame({
        'Date': forecast_dates,
        'Baseline': seg_baseline_daily
    })
    roadmap_12m['Month_Idx'] = roadmap_12m['Date'].dt.to_period('M').factorize()[0]

    # 3. Define the monthly incremental growth (Linear scaling over 12 months)
    # We calculate the maturity multiplier required to reach the final month's target
    # Based on user's sample: ~259k extra clicks in Month 12
    total_months = 12

    # Calculate daily growth factor that scales linearly by month index
    # Month 0: Factor = 1/12 of maturity, Month 11: Factor = 12/12 of maturity
    group_row = group_summary[group_summary['Page Group'] == target_group].iloc[0]
    adj = SCENARIO_ADJUSTMENTS.get(target_group, {})

    # Get the maturity multiplier from the taxonomy/CTR logic
    target_rank = adj.get('target_rank', 5.5)
    target_ctr = final_curve[(final_curve['Page Group'] == target_group) & (final_curve['Rank'] == int(round(target_rank)))]['Interpolated_CTR'].values[0]
    current_ctr = group_row['Clicks'] / group_row['Impressions']
    maturity_mult = (target_ctr / current_ctr)

    # Apply linear scaling: Growth multiplier ramps from 1.0 to maturity_mult over 365 days
    days_elapsed = np.arange(len(forecast_dates))
    linear_mults = 1.0 + (maturity_mult - 1.0) * (days_elapsed / 365)

    roadmap_12m['Roadmap_Scenario'] = roadmap_12m['Baseline'] * linear_mults

    # 4. Monthly Rollup Table
    monthly_12m = roadmap_12m.groupby(roadmap_12m['Date'].dt.to_period('M')).agg(
        Baseline=('Baseline', 'sum'),
        Roadmap=('Roadmap_Scenario', 'sum')
    ).reset_index()

    monthly_12m['Monthly_Delta'] = monthly_12m['Roadmap'] - monthly_12m['Baseline']
    monthly_12m['Cumulative_Delta'] = monthly_12m['Monthly_Delta'].cumsum()

    print(f'### 12-Month Linear Roadmap: {target_group}')
    display(monthly_12m.style.format({
        'Baseline': '{:,.0f}',
        'Roadmap': '{:,.0f}',
        'Monthly_Delta': '{:+,.0f}',
        'Cumulative_Delta': '{:,.0f}'
    }))

    # Export
    monthly_12m.to_csv('roadmap_12month_linear.csv', index=False)
    print('\n✅ Exported: roadmap_12month_linear.csv')
else:
    print(f'Segment {target_group} not found in taxonomy.')